# Real SPB Ready Pack

Пакет в стиле sandbox для тестирования алгоритмов на реальных анонимизированных данных.

В этом ноутбуке:
1. Формируем вариант датасета по доле активных машин.
2. Получаем явный слой ограничений (`derived.solver_ready`): матрицы совместимости, лимиты, расстояния.
3. Запускаем solver.
4. Смотрим классическую карту и метрики.
5. Рисуем циклы только выбранных агентов (old-style + interactive).


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import json
import sys

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "simple_solver_min.py").exists():
    NOTEBOOK_DIR = Path("data/realistic_spb_real_notebook_data").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import notebook_utils as _nbu
nbu = importlib.reload(_nbu)


## 1) Параметры и формирование варианта датасета


In [ ]:
DATA_DIR = NOTEBOOK_DIR / "data"
BASE_DATASET_PATH = DATA_DIR / "dataset_real_spb_ready.json"
SUMMARY_PATH = DATA_DIR / "summary_real_spb_ready.json"
BASE_GRAPH_PNG = DATA_DIR / "graph_real_spb_ready.png"

SEED = 42
AGENT_FRACTION = 1.0  # change in (0, 1]
DAYS = 3
MASS_NOISE = 0.08
RENDER_FIRST_DAY = True
RENDER_UTILIZATION = False

# Old-style selected agents by 1-based index in active-agent list from solution
SELECTED_AGENT_INDICES = [1, 9, 12]
SELECTED_AGENT_IDS = None  # alternative explicit ids list, e.g. ["AGENT_00001"]

TAG = nbu.fraction_tag(AGENT_FRACTION)
WORK_DATASET_PATH = DATA_DIR / f"dataset_real_spb_{TAG}.json"
OUTPUT_ROOT = NOTEBOOK_DIR / "runtime_outputs" / TAG

_ = nbu.prepare_dataset_variant(
    base_dataset_path=BASE_DATASET_PATH,
    out_dataset_path=WORK_DATASET_PATH,
    agent_fraction=AGENT_FRACTION,
    seed=SEED,
)

print(f"WORK_DATASET_PATH = {WORK_DATASET_PATH}")
print(f"OUTPUT_ROOT      = {OUTPUT_ROOT}")

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
summary


## 2) Структура solver-ready ограничений


In [ ]:
dataset_payload = json.loads(WORK_DATASET_PATH.read_text(encoding="utf-8"))
solver_ready = dataset_payload.get("derived", {}).get("solver_ready", {})

print("solver_ready keys:", solver_ready.keys())
print("sets sizes:", {k: len(v) for k, v in solver_ready.get("sets", {}).items()})
print("compatibility keys:", solver_ready.get("compatibility", {}).keys())
print("limits keys:", solver_ready.get("limits", {}).keys())
print("model constraints count:", len(solver_ready.get("model", {}).get("constraints", [])))

pd.DataFrame(solver_ready.get("limits", {}).get("agent_limits", [])).head(5)


## 3) Базовый граф


In [ ]:
img = mpimg.imread(BASE_GRAPH_PNG)
plt.figure(figsize=(14, 10))
plt.imshow(img)
plt.axis("off")
plt.title("Real SPB ready graph")
plt.show()


## 4) Запуск solver


In [ ]:
results_df = nbu.run_solver(
    notebook_dir=NOTEBOOK_DIR,
    dataset_path=WORK_DATASET_PATH,
    output_root=OUTPUT_ROOT,
    days=DAYS,
    seed=SEED,
    mass_noise=MASS_NOISE,
    render_first_day=RENDER_FIRST_DAY,
    render_utilization=RENDER_UTILIZATION,
)
results_df


## 5) Метрики по дням


In [ ]:
if "results_df" not in globals():
    results_df = nbu.load_results_df(OUTPUT_ROOT)

nbu.plot_daily_metrics(results_df)


## 6) Day 1: классическая карта + coverage


In [ ]:
DAY_INDEX = 1

classic = nbu.show_day_classic_solution(
    output_root=OUTPUT_ROOT,
    day_index=DAY_INDEX,
    title_suffix=TAG,
)
report_day1 = classic["report"]

print("Day1 constraints_check_summary:")
print(json.dumps(report_day1["constraints_check_summary"], indent=2, ensure_ascii=False))
print("\nDay 1 run metrics:")
print(classic["metrics_df"].to_string(index=False))

classic["coverage_df"]


## 7) Day 1: old-style циклы только для выбранных агентов


In [ ]:
sel = nbu.render_selected_agents_cycles_static(
    dataset_path=WORK_DATASET_PATH,
    output_root=OUTPUT_ROOT,
    day_index=DAY_INDEX,
    selected_agent_indices=SELECTED_AGENT_INDICES,
    selected_agent_ids=SELECTED_AGENT_IDS,
)

print("Selected agent ids:", sel["selected_agent_ids"])
print("Active agents index map (head):")
sel["active_agents_df"].head(20)


## 8) Day 1: интерактивная карта (масштаб + selected agents)


In [ ]:
OPEN_BROWSER = False  # switch to True for external browser tab

viz = nbu.render_day_agent_cycles(
    dataset_path=WORK_DATASET_PATH,
    output_root=OUTPUT_ROOT,
    day_index=DAY_INDEX,
    selected_agent_indices=SELECTED_AGENT_INDICES,
    selected_agent_ids=SELECTED_AGENT_IDS,
    open_browser=OPEN_BROWSER,
)

print(f"Interactive HTML map: {viz['html_path']}")
print("Selected agent ids:", viz["selected_agent_ids"])
viz["segments_df"].head(40)


## 9) Day 1: загрузка объектов


In [ ]:
pd.DataFrame(report_day1["object_capacity_summary"]["objects"])
